# Day 2(M1 Day02) 실습 — 역할 분리·데이터 계약·업무 인계

**목표**: 역할 정의서로 조사원·작성자·검토자를 나누고, 역할 사이 전달을 데이터 계약(Pydantic)으로 고정한 뒤, 요청·원문·중간 결과를 빠뜨리지 않는 업무 인계(Handoff)를 만든다.
## 오늘의 전체 그림

**오늘의 질문**: 역할을 어떻게 나누고, 무엇을 어떤 형식으로 넘겨야 빠짐없이 전달되나?

```text
[Part 1 만든다]   역할 정의서 ROLES · 조사원 → 작성자 → 검토자 (자유 텍스트 + state 딕셔너리)
      │           → 받는 쪽 코드가 사실 목록·판정을 꺼내 쓰기 어렵다
      ▼
[Part 2 바꿔보기]   1인 2역 · 파싱 성공률 · 형식 지시 vs 데이터 계약 ResearchNotes
      │           → 형식은 말이 아니라 계약(Pydantic)으로 고정한다
      ▼
[Part 3 조립한다] 계약 3종(ResearchNotes · Draft · Review)으로 run_research_team · 새 자료 재사용
                  → (확장) 인계서 Handoff로 요청·원문이 끝까지 간다
```

| Part | 블록 | 답하는 질문 | 다음 Part로 넘기는 것 |
| --- | --- | --- | --- |
| Part 1 | 따라 하기 | 역할을 나누면 각자 제 일만 하나? | `ROLES` · `role_to_system` · 자유 텍스트의 불편함 |
| Part 2 | 바꿔 보기 | 경계가 무너지면? 형식은 무엇으로 고정하나? | 데이터 계약 `ResearchNotes` |
| Part 3 | 미니 프로젝트 | 계약 팀은 새 자료에도 도나? 요청은 끝까지 가나? | `run_research_team` · `Handoff` → Day 3 State |


Day 1은 역할을 함수로 하드코딩했다 → 오늘은 역할을 **설계**한다.

## 0. 환경 준비

> LangChain은 모델 호출과 구조화 출력(`with_structured_output`)에, Pydantic은 데이터 계약에 쓴다 → 인계서를 함수가 손으로 넘기는 일을 Day 3에서 LangGraph State가 대신한다.

생성 역할(조사원·작성자)과 판정 역할(검토자)의 temperature를 다르게 둔다.

In [2]:
import os

# 추적 끄기: 2과목 Day17에서 .env의 LANGSMITH_TRACING을 켰다면 오늘 호출이 전부 전송된다
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, ValidationError

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)  # 생성 역할: 조사원·작성자
llm_strict = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 판정 역할: 검토자

print("준비 완료:", llm.model_name)

준비 완료: gpt-4o-mini


## Part 1. 따라 하기 — 역할 정의서와 자유 텍스트 팀

**전체 그림 1/3**:  세 개의 역할을 정의하고 자유 텍스트로 전달한다 

### 1-1. 모의 자료 준비

조사원이 다룰 원문이다. 외부 검색 대신 모의 자료를 쓴다.

<details>
<summary>왜 모의 자료인가</summary>

- LLM 지식으로 "조사"를 시키면 지어낸 사실(환각)이 섞여, 뭐가 맞는지 수업에서 확인할 수 없다.
- 원문을 주면 "추출 → 작성 → 대조"의 정확성을 눈으로 검증할 수 있다.
</details>

In [3]:
SOURCE = """[내부 소개 자료] 달빛 도서관
달빛 도서관은 밤 10시부터 새벽 2시까지 여는 무인 야간 도서관이다.
서울 마포구에 1호점이 있으며, 좌석은 총 48석이다.
입장은 월 구독제이며, 구독료는 월 19,900원이다.
사서 대신 안내 로봇 '루나'가 책 추천과 반납을 돕는다.
조용한 환경을 위해 통화와 취식은 금지된다.
회원의 62%가 20~30대 직장인이다.
2026년 12월에 부산 2호점이 문을 열 예정이다."""

print(SOURCE)

[내부 소개 자료] 달빛 도서관
달빛 도서관은 밤 10시부터 새벽 2시까지 여는 무인 야간 도서관이다.
서울 마포구에 1호점이 있으며, 좌석은 총 48석이다.
입장은 월 구독제이며, 구독료는 월 19,900원이다.
사서 대신 안내 로봇 '루나'가 책 추천과 반납을 돕는다.
조용한 환경을 위해 통화와 취식은 금지된다.
회원의 62%가 20~30대 직장인이다.
2026년 12월에 부산 2호점이 문을 열 예정이다.


### 1-2. 3개의 역할 정의

역할 정의서(Role Card)의 다섯 항목: **이름 · 임무 한 줄 · 입력 · 출력 · 하지 않는 일**. 

딕셔너리로 정의서를 작성하고, system 프롬프트로 바꾼다.

> 💡 "하지 않는 일"이 핵심이다.

In [5]:
ROLES = {
    "researcher": {
        "이름": "조사원(Researcher)",
        "임무": "원문에서 핵심 사실을 뽑아 정리한다",
        "입력": "원문 텍스트",
        "출력": "핵심 사실 5개 목록",
        "하지 않는 일": "원문에 없는 사실을 추측하지 않으며 소개 글을 작성하지 않는다",
        
    },
    "writer": {
        "이름": "작성자(Writer)",
        "임무": "조사원이 준 사실만 사용해 소개 글을 쓴다",
        "입력": "핵심 사실 목록",
        "출력": "제목 + 소개 단락(3~4문장)",
        "하지 않는 일": "원문에 없는 사실을 추가하지 않으며 초안의 사실 여부를 판정하지 않는다",
        
    },
    "reviewer": {
        "이름": "검토자(Reviewer)",
        "임무": "초안이 사실과 일치하는지 판정한다",
        "입력": "핵심 사실 목록 + 초안",
        "출력": "통과 여부 + 지적 사항",
        "하지 않는 일": "초안을 직접 수정하거나 새 소개 글을 작성하지 않는다",
        
    },
}


def role_to_system(role):
    """역할 정의서를 system 프롬프트로 바꾼다."""
    
    return f"""당신은 리서치 팀의 {role['이름']}입니다.
임무: {role['임무']}.
하지 않는 일: {role['하지 않는 일']}.
"""


print(role_to_system(ROLES["researcher"]))

당신은 리서치 팀의 조사원(Researcher)입니다.
임무: 원문에서 핵심 사실을 뽑아 정리한다.
하지 않는 일: 원문에 없는 사실을 추측하지 않으며 소개 글을 작성하지 않는다.



### 1-3. 조사원 :  원문 → 핵심 사실

조사원은 원문에서 사실 5개를 뽑는다. 지금은 출력이 **자유 텍스트**다.

In [6]:
def researcher_free(source):
    """조사원(자유 텍스트 버전): 원문 → 핵심 사실 5개"""
    
    # TODO: prompt를 채우세요 — 조사원 역할(role_to_system(ROLES["researcher"])) + "핵심 사실 5개를 골라 목록으로 정리" + [원문] source
    #       형식(번호·불릿)은 일부러 지정하지 않는다 → 2-2에서 파싱 성공률을 잰다
    prompt = f"""
{role_to_system(ROLES['researcher'])}

핵심 사실 5개를 골라 목록으로 정리하세요.
형식은 자유롭게 작성하되 원문에 없는 내용은 추가하지 마세요.

[원문]
{source}
""".strip()
    
    response = llm.invoke(prompt)
    return response.content.strip()

facts_text = researcher_free(SOURCE)
print(facts_text)

**관찰 포인트**: 이 셀을 여러 번 실행해 본다. 번호 목록·불릿·서문 등 형식이 실행마다 같은가? 다음 역할이 이걸 코드로 받으려면?

### 1-4. 작성자 : 딕셔너리(공유메모장)로 결과 전달

팀이 함께 쓰는 공유 메모장(state, 딕셔너리)에 결과를 담아 다음 역할에게 넘긴다.

<details>
<summary>용어 미리 풀기 — state</summary>

- **state(상태)**: 흐름을 오가며 들고 다니는 공유 메모장이다 (2과목 LangGraph의 State와 같은 개념). 오늘은 딕셔너리로 손 전달하고, Day 3에서 LangGraph가 자동으로 나른다.
</details>

In [8]:
# 공유 메모장
# TODO: state를 딕셔너리로 만드세요 — "source"에 SOURCE, "facts_text"에 1-3의 facts_text
state = {"source": SOURCE, "facts_text": facts_text}


def writer_free(state):
    """작성자(자유 텍스트 버전): 사실 목록 → 소개 글"""
    
    prompt = f"""
{role_to_system(ROLES['writer'])}

[조사원이 정리한 사실]
{state['facts_text']}

위 사실로 서비스 소개 글을 작성하세요. 본문은 3~4문장으로 쓰고 원문에 없는 사실은 추가하지 마세요.
""".strip()
    return llm.invoke(prompt).content.strip()


state["draft_text"] = writer_free(state)
print(state["draft_text"])

우리 서비스는 사용자에게 쉽고 편리한 솔루션을 제공하는 것을 목표로 합니다. 직관적인 인터페이스와 다양한 기능을 통해 사용자는 효율적으로 작업을 수행할 수 있습니다. 고객의 요구를 반영하여 지속적으로 개선하며, 최상의 사용자 경험을 보장합니다. 함께 성장하며 발전하는 서비스를 경험해 보세요.


### 1-5. 검토자 — 사실 대조 판정

검토자는 판정만 한다. 고쳐 쓰기는 검토자의 "하지 않는 일"이다.

In [9]:
def reviewer_free(state):
    """검토자(자유 텍스트 버전): 사실·초안 대조 판정"""
    
    prompt = f"""
{role_to_system(ROLES['reviewer'])}

[기준 사실]
{state['facts_text']}

[초안]
{state['draft_text']}

초안이 기준 사실을 모두 반영했는지, 기준 사실에 없는 내용을 지어내지 않았는지 판정하세요.
문제가 있으면 지적 사항을 먼저 쓰고, 마지막 줄에는 반드시 '통과' 또는 '불통과' 중 하나만 쓰세요.
""".strip()
    return llm_strict.invoke(prompt).content.strip()


state["review_text"] = reviewer_free(state)
print(state["review_text"])

초안은 서비스의 목표와 사용자 경험에 대한 설명을 포함하고 있지만, 기준 사실이 명시되어 있지 않아 초안이 사실과 일치하는지 여부를 판단할 수 없습니다. 또한, "함께 성장하며 발전하는 서비스를 경험해 보세요"와 같은 문구는 구체적인 사실에 기반하지 않고 일반적인 마케팅 문구로 보입니다. 따라서 초안이 기준 사실을 모두 반영했는지, 기준 사실에 없는 내용을 지어내지 않았는지에 대한 판단이 어렵습니다.

불통과


**관찰 — 자유 텍스트 전달의 불편함**

- 조사원의 사실 목록: 작성자 코드가 **목록으로 꺼내 쓸 수 없다** (그냥 긴 문자열)
- 검토자의 판정: "통과"인지 코드로 읽으려면 **문자열을 뒤져야** 한다
- 형식은 실행마다 흔들린다 → Part 2에서 숫자로 확인하고 계약으로 고정한다

## Part 2. 바꿔 보기 — 역할 경계와 데이터 계약

**전체 그림 2/3**: Part 1의 `ROLES`로 경계와 전달 형식을 실험한다 → 계약 `ResearchNotes`를 Part 3이 그대로 쓴다.

### 2-1. 단일 에이전트

조사와 작성을 한 에이전트에 합쳐 본다. 

In [10]:
# TODO: 역할 정의서 없이 한 에이전트에게 "조사와 작성을 모두 담당" — 원문에서 사실을 뽑고 그 사실로 소개 글도 쓰라는 prompt를 채우세요 ([원문] SOURCE 포함)
combined_prompt = f""" 당신은 조사과 작성을 모두 담당합니다.
다음 원문에서 핵심 사실을 뽑고, 그 사실로 소개 글도 쓰세요

[원문]
{SORUCE}

"""



combined_out = llm.invoke(combined_prompt).content.strip()

print(combined_out)
print()

# TODO: 아래 빈 문자열 7개를 원문에 없는 꾸밈말 후보로 채우세요 (예: 아늑, 환상, 최고, 특별, 안성맞춤, 편안한, 새로운)
# 원문에 없는 꾸밈말이 섞였는지 확인한다
for word in ["혁신", "아늑", "최고", "특별", "안성맞춤", "편안한", "새로운", "환상적인"]:
    if word in combined_out:
        print(f'[주의] 원문에 없는 표현: "{word}"')

NameError: name 'SORUCE' is not defined

**관찰 포인트**: 원문에 없는 표현이 섞였는가? 사실 목록이 따로 없으면 검토자는 무엇을 기준으로 대조해야 할까?

### 2-2. 자유 텍스트 전달 측정 — 파싱 성공률

받는 쪽(작성자) 코드가 `"- "`로 시작한다고 약속한다. 

조사원(자유 텍스트)을 5회 실행해 몇 번이나 제대로 받는지 센다.

> ⚠️ 2-2~2-4는 셀마다 LLM을 5번 호출한다.

In [ ]:
def parse_facts(text):
    """받는 쪽 코드: '- '로 시작하는 줄만 사실로 인정한다"""
    # TODO: text의 줄 중 '- '로 시작하는 줄만 골라, 앞의 '- '를 뗀 문자열 리스트로 반환하세요


ok = 0
for i in range(5):
    # TODO: 형식을 지정하지 않은 조사원(researcher_free)을 SOURCE로 실행하고, 결과를 parse_facts로 파싱해 facts에 담으세요
    facts = ""
    
    if len(facts) == 5:
        ok += 1
    print(f"{i + 1}회차: 사실 {len(facts)}개")
print("형식 미지정 파싱 성공:", ok, "/ 5")

### 2-3. 프롬프트로 형식 지시 강화

"그럼 프롬프트에 형식을 정확히 지시하면 되지 않나?" → 직접 확인한다.

In [ ]:
def researcher_strict_format(source):
    # TODO: 출력 형식을 정확히 지시하는 문장을 채우세요
    prompt = f""""""
    
    return llm.invoke(prompt).content.strip()


ok = 0
for i in range(5):
    # TODO: 형식을 지시한 조사원(researcher_strict_format)을 SOURCE로 실행하고, 결과를 parse_facts로 파싱해 facts에 담으세요
    facts = ""
    
    if len(facts) == 5:
        ok += 1
    print(f"{i + 1}회차: 사실 {len(facts)}개")
print("형식 지시 파싱 성공:", ok, "/ 5")

**관찰 포인트**: 형식 지시로 성공률이 얼마나 올랐는가? 그런데 프롬프트를 누가 고치면, 받는 쪽 파서와 짝이 맞는지 누가 검사해 줄까?

### 2-4. 출력 형식 계약 도입 — Pydantic으로 고정

형식을 **코드로** 약속한다 (2과목 Day 3의 `with_structured_output` 재사용).

In [12]:
class ResearchNotes(BaseModel):
    """조사원의 출력 계약"""

    facts: list[str] = Field(description="원문에서 뽑은 핵심 사실 5개. 각 사실은 한 문장.")



def researcher_contract(source):
    prompt = f"""
{role_to_system(ROLES['researcher'])}

핵심 사실 5개를 뽑으세요. 각 사실은 원문에 근거한 한 문장으로 작성하세요.

[원문]
{source}
""".strip()
    return llm.with_structured_output(ResearchNotes).invoke(prompt)


ok = 0
for i in range(5):
    notes = researcher_contract(SOURCE)
    if len(notes.facts) == 5:
        ok += 1
    print(f"{i + 1}회차: facts {len(notes.facts)}개")
print("계약 성공:", ok, "/ 5")

1회차: facts 5개
2회차: facts 5개
3회차: facts 5개
4회차: facts 5개
5회차: facts 5개
계약 성공: 5 / 5


### 2-5. 계약 위반 즉시 예외, 필드 설명이 지시로 작동

계약 위반을 확인한다.

In [ ]:
# 성질 1: 계약에 어긋난 데이터는 그 자리에서 예외(ValidationError)를 낸다
class StrictNotes(BaseModel):
    facts: list[str] = Field(min_length=5, max_length=5, description="핵심 사실 정확히 5개")


try:
    StrictNotes(facts=["사실이 두 개뿐", "이라면"])
except ValidationError as e:
    print("계약 위반이 즉시 드러난다:")
    print(e)

In [ ]:
# 성질 2: 계약(스키마)은 그 자체가 지시다 — 프롬프트를 안 바꿔도 새 필드가 채워진다
class NotesWithCategory(BaseModel):
    # TODO: 두 Field에 description을 채우세요 — facts: "원문에서 뽑은 핵심 사실 5개",
    #       category: "자료의 분류 (예: 서비스 소개, 행사 안내, 채용 공고)"
    facts: list[str] = Field()
    category: str = Field()

# TODO: llm에 NotesWithCategory 계약을 걸어(with_structured_output) SOURCE에서 "핵심 사실 5개를 뽑으라"고만 지시해 결과를 res에 담으세요
#       분류 요청은 프롬프트에 넣지 않는다 → category description만 보고 채워지는지 확인
res = ""

print("프롬프트에 없던 요구가 계약만으로 채워졌다 → 분류:", res.category)

**관찰 포인트**: 형식 미지정·형식 지시·계약의 파싱 성공률을 비교한다. 형식 지시(말 약속)와 계약(코드 약속)의 결정적 차이는 무엇인가?

> ⚠️ 계약과 프롬프트를 **충돌**시키면 안 된다 (예: 프롬프트 "사실 3개만" + 계약 "정확히 5개"). 사전 검증에서 모델이 헤매며 토큰 한도까지 폭주한 실행이 있었다 → 계약과 프롬프트는 항상 일치시킨다.

## Part 3. 미니 프로젝트 — 계약이 적용된 3-역할 리서치 팀

**전체 그림 3/3**: Part 2의 계약에 `Draft` · `Review`를 더해 팀을 완성한다 → 확장에서 인계서로 요청·원문까지 넘긴다.

모든 역할 사이 전달을 데이터 계약으로 바꿔 파이프라인을 완성한다.

| 흐름 | 계약 |
| --- | --- |
| 원문 → 조사원 | `ResearchNotes` (facts) |
| 사실 → 작성자 | `Draft` (title, body) |
| 사실+초안 → 검토자 | `Review` (passed, missing_facts, comments) |

### 3-1. 출력 형식 - 계약 3종

역할 정의서의 "출력" 항목을 그대로 계약 클래스로 옮긴다.
조사원의 계약 `ResearchNotes`는 2-4에서 만들었다 → 이 셀에서는 `Draft` · `Review` 두 개만 더한다.

In [13]:
class Draft(BaseModel):
    """작성자의 출력 계약"""

    # TODO: 두 Field에 description을 채우세요 — title: "소개 글 제목", body: "사실에 근거한 소개 단락 (3~4문장)"
    title: str = Field(description="소개 글 제목")
    body: str = Field(description="사실에 근거한 소개 단락 (3~4문장)")


class Review(BaseModel):
    """검토자의 출력 계약"""

    # TODO: passed 필드를 만드세요 — True/False 타입, description은 "모든 사실이 정확히 반영되고 지어낸 내용이 없으면 True"
    passed: bool = Field(description="모든 사실이 정확히 반영되고 지어낸 내용이 없으면 True")
    missing_facts: list[str] = Field()
    comments: str = Field()


print("계약 준비 완료: ResearchNotes(2-4) · Draft · Review")

계약 준비 완료: ResearchNotes(2-4) · Draft · Review


### 3-2. 역할 에이전트 3종 (계약 버전)

출력을 고정한다.

In [14]:
def researcher_agent(source):
    """조사원(계약 버전): 원문 → ResearchNotes"""
    prompt = f"""
{role_to_system(ROLES['researcher'])}

핵심 사실 5개를 뽑으세요.

[원문]
{source}
""".strip()
    return llm.with_structured_output(ResearchNotes).invoke(prompt)


def writer_agent(facts):
    """작성자(계약 버전): 사실 리스트 → Draft"""
    facts_text = "\n".join(f"- {f}" for f in facts)
    prompt = f"""
{role_to_system(ROLES['writer'])}

[조사원이 정리한 사실]
{facts_text}

위 사실로 서비스 소개 글을 작성하세요. 본문은 3~4문장으로 쓰고 사실에 없는 내용은 추가하지 마세요.
""".strip()
    return llm.with_structured_output(Draft).invoke(prompt)

In [15]:
def reviewer_agent(facts, draft):
    """검토자(계약 버전): 사실+초안 → Review"""
    facts_text = "\n".join(f"- {f}" for f in facts)
    
    prompt = f"""
{role_to_system(ROLES['reviewer'])}

[기준 사실]
{facts_text}

[초안 제목]
{draft.title}

[초안 본문]
{draft.body}

각 사실이 초안에 반영됐는지, 사실에 없는 내용이 섞였는지 판정하세요.
""".strip()
    return llm_strict.with_structured_output(Review).invoke(prompt)


def run_research_team(source):
    """조사 → 작성 → 검토 파이프라인 (계약 적용)"""
    
    notes = researcher_agent(source)
    draft = writer_agent(notes.facts)
    review = reviewer_agent(notes.facts, draft)
    return notes, draft, review

### 3-3. 파이프라인 실행

각 단계의 출력이 계약 객체라 `.facts`, `.title`, `.passed`로 바로 꺼내 쓴다.

In [17]:
notes, draft, review = run_research_team(SOURCE)

print("[조사원의 사실]")
for f in notes.facts:
    print("-", f)
print("\n[작성자의 초안]")
print("제목:", draft.title)
print("본문:", draft.body)
print("\n[검토자의 판정]")
print("통과:", review.passed)
print("빠진 사실:", review.missing_facts)
print("총평:", review.comments)

[조사원의 사실]
- 달빛 도서관은 밤 10시부터 새벽 2시까지 여는 무인 야간 도서관이다.
- 서울 마포구에 1호점이 있으며, 좌석은 총 48석이다.
- 입장은 월 구독제이며, 구독료는 월 19,900원이다.
- 사서 대신 안내 로봇 '루나'가 책 추천과 반납을 돕는다.
- 조용한 환경을 위해 통화와 취식은 금지된다.

[작성자의 초안]
제목: 달빛 도서관 소개
본문: 달빛 도서관은 서울 마포구에 위치한 무인 야간 도서관으로, 밤 10시부터 새벽 2시까지 운영됩니다. 총 48석의 좌석을 마련하여 이용자들에게 아늑한 독서 환경을 제공합니다. 입장은 월 구독제로, 구독료는 19,900원입니다. 이곳에서는 사서 대신 안내 로봇 '루나'가 책 추천과 반납을 도와주며, 조용한 환경 유지를 위해 통화와 취식은 금지되고 있습니다.

[검토자의 판정]
통과: True
빠진 사실: []
총평: 


**관찰 포인트**: 원문(1-1)은 8줄인데 조사원은 사실 5개만 넘겼다. 원문에서 빠진 사실이 있는가? 검토자는 그것을 알아챘는가?

### 3-4. 새 자료로 재사용

성격이 다른 자료로 바꿔 실행한다. 역할·계약 설계가 잘 됐다면 코드는 한 줄도 바꿀 필요가 없다.

In [19]:
SOURCE2 = """[내부 소개 자료] 새벽 수영 클럽
새벽 수영 클럽은 평일 오전 5시 30분부터 7시까지 운영하는 직장인 수영 모임이다.
장소는 성수동의 실내 수영장이며, 레인 3개를 통째로 빌려 쓴다.
월 회비는 89,000원이고, 강습료가 포함되어 있다.
초급반·중급반·자유수영반 세 반으로 나뉜다.
수료 기준을 통과하면 한강 야외 수영 행사에 참가할 수 있다.
현재 회원은 74명이며 대기자가 30명이 넘는다."""

notes2, draft2, review2 = run_research_team(SOURCE2)
print("[제목]", draft2.title)
print("[본문]", draft2.body)
print("[판정]", review2.passed, "/", review2.comments)

[제목] 새벽 수영 클럽 소개
[본문] 새벽 수영 클럽은 성수동의 실내 수영장에서 평일 오전 5시 30분부터 7시까지 운영됩니다. 레인 3개를 통째로 빌려 사용하며, 월 회비는 89,000원으로 강습료가 포함되어 있습니다. 수업은 초급반, 중급반, 자유수영반으로 나뉘어 진행되며, 현재 74명의 회원과 30명이 넘는 대기자가 있습니다.
[판정] True / 


## Part 3-확장. 업무 인계(Handoff) — 빠뜨리지 않고 넘기기

지금 팀은 **자기 앞 사람이 준 것만** 받는다. 함수 시그니처가 그대로 보여 준다.

- `writer_agent(facts)` — 사용자 요청이 들어갈 자리가 없다 → 요청은 작성자에게 갈 수 없다.
- `reviewer_agent(facts, draft)` — 원문이 들어갈 자리가 없다 → 3-3에서 검토자가 빠진 사실을 못 잡은 이유다.

> 💡 2과목 LangGraph로 치면 `request`·`source` 키를 State에 두는 것과 같다. 인계서는 여기에 세 가지를 더한다: **칸마다 채우는 역할**, **받을 때 빈 칸 점검**, 누가 무엇을 했는지 남기는 **이력(`log`)**.

<details>
<summary>용어 미리 풀기 — 업무 인계(Handoff)</summary>

- **업무 인계(Handoff)**: 한 역할이 일을 다음 역할에 넘기는 것이다. 사람 팀의 인수인계서처럼, 다음 사람이 일하는 데 필요한 것(요청·자료·중간 결과·지금까지 한 일)을 **빠짐없이** 넘겨야 한다.
- 3과목 단위 프로젝트 평가 항목에 "Handoff 시 컨텍스트(상태·메모리·중간 결과)가 누락 없이 전달되는 구조"가 있다 → 오늘 만드는 인계서가 그 출발점이다.
</details>

### 3-5. 인계서 — 모든 역할이 같은 문서를 채워 넘긴다

인계서(`Handoff`) 하나에 요청·원문·중간 결과·이력을 담는다. 각 역할은 인계서를 받아 **자기 칸만 채우고** 다음 역할에게 넘긴다.

| 칸 | 채우는 역할 | 읽는 역할 |
| --- | --- | --- |
| `request`, `source` | 시작할 때 | 모두 |
| `facts` | 조사원 | 작성자·검토자 |
| `title`, `body` | 작성자 | 검토자 |
| `passed`, `comments` | 검토자 | 사람 |
| `log` | 모두 (한 줄씩) | 사람 |

In [ ]:
REQUEST = "대학생에게 소개하는 글로 써 주세요. '시험 기간'이라는 표현을 꼭 넣어 주세요."


class Handoff(BaseModel):
    """역할 사이에 넘기는 인계서. 각 역할이 자기 칸을 채워 다음 역할에게 넘긴다."""

    request: str  # 사용자 요청 — 처음부터 끝까지 유지
    source: str  # 원문 — 검토자까지 전달
    
    facts: list[str] = Field(default_factory=list)
    title: str = ""  # 작성자가 채움
    body: str = ""
    passed: bool = False  # 검토자가 채움
    comments: str = ""
    log: list[str] = []  # 누가 무엇을 했는지 이력


h = Handoff(request=REQUEST, source=SOURCE)
print("빈 인계서:", h.facts, h.title, h.log)

각 역할을 "인계서를 받아 → 자기 칸을 채우고 → 돌려주는" 함수로 바꾼다.


In [ ]:
def researcher_step(h):
    """조사원: 요청을 참고해 원문에서 사실 5개를 뽑아 facts 칸을 채운다."""
    
    # TODO: prompt를 채우세요 — 조사원 역할 + [사용자 요청] h.request + "요청에 도움이 되는 핵심 사실 5개를 뽑으세요" + [원문] h.source
    prompt = f""""""
    
    notes = llm.with_structured_output(ResearchNotes).invoke(prompt)
    h.facts = notes.facts
    h.log.append(f"조사원: 사실 {len(h.facts)}개 전달")
    return h


def writer_step(h):
    """작성자: 인계서를 점검한 뒤 요청과 사실로 초안을 써서 title·body 칸을 채운다."""
    
    # TODO: 앞 역할(조사원)이 채워야 할 facts 칸이 비어 있으면
    #       ValueError("인계 오류: 조사원이 채울 facts 칸이 비어 있습니다")를 일으키세요
    
    
    facts_text = "\n".join(f"- {f}" for f in h.facts)
    
    # TODO: prompt를 채우세요 — 작성자 역할 + [사용자 요청] h.request + [조사원이 정리한 사실] facts_text
    #       + "요청에 맞춰 소개 글을 작성하세요. 본문은 3~4문장."
    prompt = f""""""
    
    draft = llm.with_structured_output(Draft).invoke(prompt)
    h.title = draft.title
    h.body = draft.body
    h.log.append("작성자: 초안 작성")
    return h

In [ ]:
def reviewer_step(h):
    """검토자: 요청·원문·사실·초안을 모두 보고 판정해 passed·comments 칸을 채운다."""
    facts_text = "\n".join(f"- {f}" for f in h.facts)
    
    # TODO: prompt를 채우세요 — 검토자 역할 + [사용자 요청] h.request + [원문] h.source + [조사원이 정리한 사실] facts_text
    #       + [초안] h.title·h.body + "원문과 일치하는지, 사용자 요청이 반영됐는지 판정하세요"
    prompt = f""""""
    
    review = llm_strict.with_structured_output(Review).invoke(prompt)
    h.passed = review.passed
    h.comments = review.comments
    h.log.append(f"검토자: 통과={h.passed}")
    return h


def run_team_with_handoff(request, source):
    """인계서 하나를 조사원 → 작성자 → 검토자 순서로 넘긴다."""
    h = Handoff(request=request, source=source)
    
    # TODO: 인계서 h를 조사원 → 작성자 → 검토자 단계 함수에 차례로 넘기세요 (각 단계는 h를 받아 h를 돌려준다)
    
    return h

### 3-6. 실행 — 인계서로 요청이 끝까지 간다

사용자 요청을 인계서에 담아 3번 실행한다. 작성자가 요청을 받으니 '시험 기간'이 본문에 들어가야 한다.

In [ ]:
new_hits = 0
for i in range(3):
    h = run_team_with_handoff(REQUEST, SOURCE)
    if "시험 기간" in h.body:
        new_hits += 1
    print(f"[{i + 1}회] {h.body[:60]}...")

print("요청 반영('시험 기간' 포함):", new_hits, "/ 3")


print("\n[마지막 실행의 이력]")
for line in h.log:
    print("-", line)
print("검토 총평:", h.comments)

**관찰 포인트**: 요청이 반영된 횟수는? 이력(`log`)만 보고 누가 무엇을 했는지 알 수 있는가?

### 3-7. 인계 점검 — 빈 칸은 넘기지 않는다

조사원을 건너뛰고 작성자에게 바로 넘기면 어떻게 될까? 작성자의 점검이 막아 주는지 확인한다.

In [ ]:
empty = Handoff(request=REQUEST, source=SOURCE)  # 조사원이 일하지 않은 인계서

try:
    writer_step(empty)
except ValueError as e:
    print("작성자가 일을 멈췄다 →", e)

**관찰 포인트**: 점검이 없었다면 작성자는 사실 없이 무엇을 썼을까? → 빈 칸을 그대로 넘기면 **그럴듯한 지어낸 글**이 나온다. 인계는 "넘기는 것"과 "받을 때 확인하는 것"이 한 쌍이다.

## 확인 문제

1. 역할 정의서의 다섯 항목 중 역할 경계를 지키는 데 가장 중요한 것은 무엇이고, 왜인가?
2. 형식 미지정·형식 지시·데이터 계약의 파싱 성공률은 각각 몇이었나?
3. 계약 위반이 그 자리에서 예외를 내는 것이 왜 좋은가?
4. 자료를 바꿔도 파이프라인 코드를 안 바꿔도 되는 이유는?
5. 인계서(Handoff)는 LangGraph State에 요청·원문 키를 두는 것에 무엇을 더하나?
6. 인계서(Handoff)에 담아야 하는 네 가지와, 받는 쪽이 해야 할 일은?

## 정리·회고

오늘 배운 것을 3줄로 정리해 본다.

1. 역할을 나눌 때 "하지 않는 일"을 적어야 하는 이유는?
2. 자유 텍스트 대신 데이터 계약으로 넘기면 무엇이 달라졌는가? 숫자로 답한다.
3. 인계서가 없을 때 사라진 정보는 무엇이었는가?

| 오늘 만든 것 | 다음에 쓰는 곳 |
| --- | --- |
| 역할 정의서 3종 | Day 3 Supervisor가 일을 나눠 줄 작업자 목록 |
| 데이터 계약 3종 | Day 4 병렬 결과 통합, Day 5 검증 루프(`passed`로 분기) |
| 인계서 `Handoff` | Day 3 LangGraph State — 인계서를 그래프가 자동으로 나른다 · 단위 프로젝트 `Incident` |

작성한 요약과 오늘 코드를 커밋한다.

```powershell
git add .
git commit -m "Day2: 역할 정의서 + 데이터 계약 + 업무 인계서"
```

**내일 예고**: 오늘 흐름은 조사 → 작성 → 검토로 **고정**되어 있다. Day 3에서는 관리자(Supervisor)가 상황을 보고 "다음은 누가 일할지"를 정한다 — LangGraph로 구현하고, 오늘의 인계서가 그래프의 State가 된다.